This notebook demonstrate how to create a more interesting paleo-map using GPlately.

In [ ]:
import warnings, os
import cartopy.crs as ccrs
import matplotlib.pyplot as plt

from gplately import PlateModelManager, Raster, auxiliary
from gplately.mapping.gmt_cpt import get_cmap_from_gmt_cpt

# use the auxiliary function to create a PlotTopologies instance
gplot = auxiliary.get_gplot("Muller2019", time=100)  # 100 Ma

# download the age grid at the reconstruction time
agegrid = Raster(data=PlateModelManager().get_model("Muller2019").get_raster("AgeGrids", int(gplot.time)))

fig = plt.figure(figsize=(8, 4))
ax = fig.add_subplot(111, projection=ccrs.Mollweide(190))

# download agegrid CPT file
cpt_file='agegrid.cpt'
if not os.path.isfile(cpt_file):
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/GPlates/gplately/refs/heads/master/tests-dir/unittest/create-age-grids-video/agegrid.cpt",
        cpt_file,
    )

# plot something for fun
im = gplot.plot_grid(ax, agegrid.data, cmap=get_cmap_from_gmt_cpt(cpt_file), vmin=0, vmax=200)
gplot.plot_continents(ax, facecolor="0.8")
gplot.plot_coastlines(ax, color="0.5")
with warnings.catch_warnings():
    warnings.filterwarnings('ignore', category=UserWarning)
    gplot.plot_all_topological_sections(
        ax,
        plot_subduction_teeth=True,
        other_kwargs={"color": "grey", "linewidth": 0.8},
        ridge_kwargs={"color": "black", "linewidth": 1.0},
        transform_kwargs={"color": "green", "linewidth": 1.0},
        trench_kwargs={"color": "blue", "linewidth": 1.0},
    )
gplot.plot_plate_motion_vectors(ax, spacingX=10, spacingY=10, normalise=True, zorder=10, alpha=0.9)

fig.colorbar(im, orientation="horizontal", shrink=0.4, pad=0.05, label="Age (Ma)")
plt.title(f"{int(gplot.time)} Ma")

plt.show()
